### Building a Chatbot With Multiple Tools Using LangGraph

#### Aim
Create a powerful, multi-tool conversational assistant with:
- **Arxiv Search**: To retrieve and summarize scientific research papers.
- **Wikipedia Search**: To look up encyclopedia knowledge.
- **Custom Tools**: For mathematical calculations or search.
- **ReAct Agent Architecture**: Allowing the model to call tools, receive their outputs, and synthesize a complete natural language response.


In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY", "")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY", "")
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY", "")


#### 1. Setting Up Community Tools (Arxiv & Wikipedia)


In [ ]:
from langchain_community.tools import ArxivQueryRun, WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper, ArxivAPIWrapper
from langchain_core.tools import tool

# Arxiv Tool
api_wrapper_arxiv = ArxivAPIWrapper(top_k_results=2, doc_content_chars_max=500)
arxiv = ArxivQueryRun(api_wrapper=api_wrapper_arxiv)
print(f"Tool initialized: {arxiv.name}")

# Wikipedia Tool
api_wrapper_wiki = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=500)
wiki = WikipediaQueryRun(api_wrapper=api_wrapper_wiki)
print(f"Tool initialized: {wiki.name}")


#### 2. Defining Custom Function Tools
We can also create custom tools using the `@tool` decorator.


In [ ]:
@tool
def calculate_expression(expression: str) -> str:
    """Safely evaluates a basic mathematical expression string like '2 + 2' or '10 * 5'.

    Args:
        expression (str): The mathematical expression to evaluate.

    Returns:
        str: The result of the calculation.
    """
    try:
        allowed_names = {"__builtins__": None}
        return str(eval(expression, allowed_names, {}))
    except Exception as e:
        return f"Error evaluating expression: {e}"

# Combine all tools
tools = [arxiv, wiki, calculate_expression]


#### 3. Initializing and Binding Tools to the LLM


In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(model="llama-3.3-70b-versatile")
# Alternatively with OpenAI:
# from langchain_openai import ChatOpenAI
# llm = ChatOpenAI(model="gpt-4o-mini")

llm_with_tools = llm.bind_tools(tools)


#### 4. Defining State Schema with Reducer


In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langchain_core.messages import AnyMessage, HumanMessage
from langgraph.graph.message import add_messages

class State(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]


#### 5. Building the ReAct Chatbot Graph

In a complete ReAct (Reason + Act) loop:
1. `tool_calling_llm` receives the messages and decides whether to call a tool or reply directly.
2. `tools_condition` checks if any tool calls were returned:
   - If **yes**: Routes to the `tools` node.
   - If **no**: Routes to `END`.
3. From `tools`, the edge connects back to `tool_calling_llm` so the model can read the `ToolMessage` result and produce the final summarized response for the user!


In [ ]:
from IPython.display import Image, display
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode, tools_condition

## Node definition
def tool_calling_llm(state: State) -> dict:
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

## Build Graph
builder = StateGraph(State)

builder.add_node("tool_calling_llm", tool_calling_llm)
builder.add_node("tools", ToolNode(tools))

builder.add_edge(START, "tool_calling_llm")
builder.add_conditional_edges(
    "tool_calling_llm",
    tools_condition,
)
# Loop back to tool_calling_llm to let the LLM synthesize tool results
builder.add_edge("tools", "tool_calling_llm")

graph = builder.compile()

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    print(graph.get_graph().draw_mermaid())


#### Testing the Multi-Tool Chatbot

##### 1. Querying Arxiv


In [ ]:
query_arxiv = HumanMessage(content="Summarize the core idea of the paper 1706.03762 (Attention Is All You Need).")
result_arxiv = graph.invoke({"messages": [query_arxiv]})

for m in result_arxiv['messages']:
    m.pretty_print()


##### 2. Querying Wikipedia


In [ ]:
query_wiki = HumanMessage(content="What is Machine Learning according to Wikipedia?")
result_wiki = graph.invoke({"messages": [query_wiki]})

for m in result_wiki['messages']:
    m.pretty_print()


##### 3. Querying Custom Calculation Tool


In [ ]:
query_calc = HumanMessage(content="Calculate 4523 * 87 using your calculation tool.")
result_calc = graph.invoke({"messages": [query_calc]})

for m in result_calc['messages']:
    m.pretty_print()


#### High-Level Alternative: `create_react_agent`
LangGraph also provides a prebuilt convenience helper `create_react_agent` which sets up the same State, nodes, and conditional edges in a single line:

```python
from langgraph.prebuilt import create_react_agent

prebuilt_agent = create_react_agent(llm, tools)
# response = prebuilt_agent.invoke({"messages": [HumanMessage(content="What is Machine Learning?")]})
```
